<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/concept_direction_latent_dynamics.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Concept-Direction Latent Dynamics

Where along the forward pass does a concept direction live? This notebook builds the color-vs-fruit concept direction
of the [concept-direction steering notebook](ct_concept_steering_demo.ipynb) (the `orange` sense-disambiguation
example) from each state `gemma-3-1b-it` computes on the way to its answer, and compares those directions stage by
stage:

- **Embedding stages** build the direction from token embeddings: the two target tokens (`Fruit`, `Color`), the mean
  embeddings of the concept words, and their paired rejection.
- **Answer-state stages** build it from the residual stream at the answer position of a set of probe prompts, one
  group per sense.
- **Context-state stages** build it from the residual stream at the concept word in context, its projection onto the
  answer states, and their paired residuals.

Each stage's direction is then examined four ways: how the directions align with each other, what each one does when
added after the final norm (a readout of its alignment with the unembedding), what each does when added mid-model
(causal steering), and which transcoder features its attribution graph selects. A final section projects the
per-example states and the stage directions into two dimensions.

The analysis itself lives in the `speediedan/concept-direction` Hub component
(`exp/analysis/concept_direction_latent_dynamics.py`), fetched at a pinned revision. This notebook runs it once and
lays out its report.

**Requirements:** a bf16 CUDA GPU (about 4.5 GB peak), the gated `google/gemma-3-1b-it` weights, the `examples` extra
(`plotly`, `umap-learn`) and the `git-deps` group (circuit-tracer).

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
# -- The experiment component ---------------------------------------------------------------------
# Pinned, not floating: the revision you read is the revision that executes.
COMPONENT_REPO = "speediedan/concept-direction"
COMPONENT_REVISION = "f454a0312acad29db7f7b3743f50c719f8d7a595"
EXPERIMENT_KEY = "gemma3_1b_it_local_latent_dynamics_color_fruit_orange_fs_l10_n5"
# False resolves the revision from the local components cache only: offline, after an earlier fetch.
FETCH_COMPONENT = True
# The analysis is Python that component publishes, run in this kernel. Left False, the fetch cell stops with the
# reason and where the fetched code sits, rather than opting in on your behalf.
TRUST_COMPONENT_CODE = False
# Rows shown in the top-features table for each stage.
TOP_FEATURES_SHOWN = 5

## 1. Fetch the experiment, then run it

`it.hub.pull_experiment` fetches one experiment definition and the files it owns, pinned to one commit, and returns
the snapshot directory those files sit in (`it.hub.load_experiment` resolves the same from the local cache, without
the network). Nothing executes until the trust decision below; the analysis then builds
its own session from the experiment configuration (model, transcoders, prompts, stages), so this cell is the whole
run. Expect a few minutes on a single GPU: every stage builds a direction, sweeps two steering sites and computes an
attribution graph. Its log is long, so the rendered page collapses it; the sections below lay out the report.

In [4]:
# @title 1: Fetch and run { display-mode: "form" }
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it

for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

if FETCH_COMPONENT:
    canonical_key, _, snapshot_dir = it.hub.pull_experiment(COMPONENT_REPO, EXPERIMENT_KEY, revision=COMPONENT_REVISION)
else:
    canonical_key, _, snapshot_dir = it.hub.load_experiment(COMPONENT_REPO, EXPERIMENT_KEY, revision=COMPONENT_REVISION)
print(f"fetched {COMPONENT_REPO}:{canonical_key} into {snapshot_dir.name}")

if not TRUST_COMPONENT_CODE:
    raise SystemExit(
        "Set TRUST_COMPONENT_CODE = True to continue.\n"
        f"The experiment is already cached at {snapshot_dir}, so you can read "
        "exp/analysis/concept_direction_latent_dynamics.py before deciding."
    )
os.environ["IT_TRUST_REMOTE_CODE"] = "1"
sys.path.insert(0, str(snapshot_dir))

from exp.analysis.concept_direction_latent_dynamics import run_latent_dynamics_analysis  # noqa: E402

# The fetch checks that the configuration's filename matches its key, so the key names exactly one file.
config_paths = sorted(snapshot_dir.rglob(f"{canonical_key}.yaml"))
assert len(config_paths) == 1, f"expected one {canonical_key}.yaml in the snapshot, found {config_paths}"
report = run_latent_dynamics_analysis(config_paths[0])
print(f"{len(report['stages'])} stages, {len(report['examples'])} probe examples")
print(f"store-state reconstruction check, largest difference: {report['reconstruction_max_abs_diff']:.3g}")

src/interpretune/hub/trust.py:96: Refusing to execute analysis ops from 'speediedan/jlens_steering_ops': loading it runs code published by that repo inside this process, with your filesystem, network and Hugging Face token. Interpretune does not execute hub-resident code unless you opt in.
  Inspect it first: interpretune.hub.pull('speediedan/jlens_steering_ops') caches the repo without executing anything, so you can read the entrypoint the manifest names.
  Then opt in for this session: os.environ['IT_TRUST_REMOTE_CODE'] = '1' (or export IT_TRUST_REMOTE_CODE=1 before starting the process).
  Pin a revision when you pull so trusted code cannot change under you.
  Details: https://interpretune.org/en/latest/usage/hub_trust_posture.html


fetched speediedan/concept-direction:gemma3_1b_it_local_latent_dynamics_color_fruit_orange_fs_l10_n5 into f454a0312acad29db7f7b3743f50c719f8d7a595


~/.cache/huggingface/interpretune/hub/components/models--speediedan--concept-direction/snapshots/f454a0312acad29db7f7b3743f50c719f8d7a595/exp/concept_direction.py:887: Falling back to public Neuronpedia because the local Neuronpedia webapp or DB is unavailable.


src/interpretune/config/shared.py:281: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'nnsight_cfg': NNsightConfig(model_name='google/gemma-3-1b-it', device_map='cuda', torch_dtype='float32', dispatch=True, tokenizer_kwargs={'padding_side': 'left', 'add_bos_token': True}, trust_remote_code=False, attn_implementation=None, default_padding_side='left', remote=False, api_key=None), 'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name='google/gemma-3-1b-it', transcoder_set='mwhanna/gemma-scope-2-1b-it/transcoder_all/width_16k_l0_small_affine', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=64, max_feature_nodes=None, offload=None, freeze_attention=True, freeze_norms=True, lazy_encoder=None, lazy_decoder=True, lazy_delta_rank=None, tune_base=False, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analy

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (723 outputs left out)

Attribution completed in 11.18s


~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


9 stages, 6 probe examples
store-state reconstruction check, largest difference: 0


~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 2. The prompts, the stages and the probe examples

The concept direction is built from probe prompts: one classification question, asked of each word in two groups (the
fruits and the colors), each with its expected answer. The analysis prompt the direction is then steered on is the
`orange` question, which the model answers `Color`.

Every stage is one concept direction: its **basis** (token embeddings, or residual-stream store states), its **mode**
(a plain mean difference between the two groups, or a paired rejection that removes what each pair shares), and, for
the store stages, the **raw norm** of the group separation before it is normalized.

The example table is the sanity check behind everything else: each probe prompt's expected answer, the model's margin
for it (the expected answer's logit minus the other target's), and the weight the example carries. The store keeps
only the examples the model answers as expected (`store_concept_correct_only: true` in this configuration) and weights
each by its margin; an example answered the other way would otherwise pull the example-weighted stages toward the
wrong sense, with a weight as large as its error. The `unweighted` stage counts every kept example equally.

In [5]:
# @title 2: Prompts, stage and example tables { display-mode: "form" }
from exp.analysis.concept_direction_analysis import build_classification_prompt_text
from exp.concept_direction import load_concept_pair
from interpretune.harness.config import load_experiment_config

from it_examples.utils.nb_ui_utils import (
    NEGATIVE_COLOUR,
    POSITIVE_COLOUR,
    SIGN_LEGEND,
    Signed,
    display_notebook_table,
    display_plotly_figure,
)

payload = load_experiment_config(config_paths[0])
# the concept pair is a sibling of the experiment configuration in the snapshot
pair = load_concept_pair(config_paths[0].parent / payload["EXPERIMENT"]["concept_pair_config_path"])
display_notebook_table(
    [
        (group, word, answer, build_classification_prompt_text(word, pair.classification_question))
        for group, entities in ((pair.group_a_name, pair.group_a_entities), (pair.group_b_name, pair.group_b_entities))
        for word, answer in entities
    ],
    ["group", "word", "expected answer", "probe prompt"],
    title=f"The probe prompts ({pair.concept_label})",
    caption=f"Each is rendered with the chat template. The analysis prompt: {payload['PROMPT']['text']!r}",
)

display_notebook_table(
    [
        (name, rec["basis"], rec["mode"], "" if rec.get("raw_norm") is None else f"{rec['raw_norm']:.1f}")
        for name, rec in report["stages"].items()
    ],
    ["stage", "basis", "mode", "raw norm"],
    title="The stages",
    caption="Embedding stages have no store separation, so no raw norm. Store-state reconstruction check, largest "
    f"difference: {report['reconstruction_max_abs_diff']:.3g}.",
)

examples = report["examples"]
wrong = [ex for ex in examples if not ex["answered_as_expected"]]
display_notebook_table(
    [
        (
            ex["example_index"],
            ex["group"],
            ex["expected_answer"],
            Signed(ex["expected_answer_margin"]),
            "yes" if ex["answered_as_expected"] else "no",
            f"{ex['weight']:.2f}",
            f"{ex['answer_context_cosine']:.3f}",
        )
        for ex in examples
    ],
    ["example", "group", "expected answer", "margin", "answered as expected", "weight", "answer/context cosine"],
    title=f"{len(examples) - len(wrong)} of {len(examples)} probe examples answered as expected",
    caption="The store's examples: the margin is the expected answer's logit minus the other target's.",
    legend=SIGN_LEGEND,
    emphasize=[i for i, ex in enumerate(examples) if not ex["answered_as_expected"]],
)

group,word,expected answer,probe prompt
Fruit,apple,Fruit,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". apple :"
Fruit,banana,Fruit,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". banana :"
Fruit,grape,Fruit,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". grape :"
Fruit,peach,Fruit,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". peach :"
Color,red,Color,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". red :"
Color,blue,Color,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". blue :"
Color,green,Color,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". green :"
Color,yellow,Color,"Is this a fruit or a color? Answer with one word: ""Fruit"" or ""Color"". yellow :"


stage,basis,mode,raw norm
target_token_embed_difference,embed,mean_difference,
embed_mean_difference,embed,mean_difference,
answer_state_mean_difference,store,mean_difference,59.7
context_state_mean_difference,store,mean_difference,74.8
projected_context_mean_difference,store,mean_difference,30.2
paired_residual_unweighted,store,paired_rejection,28.1
store_context_paired_rejection,store,paired_rejection,28.7
embed_paired_rejection,embed,paired_rejection,
store_answer_paired_rejection,store,paired_rejection,59.2


example,group,expected answer,margin,answered as expected,weight,answer/context cosine
0,Fruit,Fruit,+19.25,yes,19.25,0.315
1,Fruit,Fruit,+17.38,yes,17.38,0.280
2,Fruit,Fruit,+15.00,yes,15.00,0.222
3,Fruit,Fruit,+15.25,yes,15.25,0.309
4,Color,Color,+15.38,yes,15.38,0.291
5,Color,Color,+1.00,yes,1.00,0.276


## 3. How the stage directions align

Cosine similarity between every pair of stage directions, all in the residual stream's coordinates. Blocks along the
diagonal are stages that agree on the concept; near-zero entries between blocks mean two stages read the same concept
along directions that have nothing in common.

In [6]:
# @title 3: Stage cosine matrix { display-mode: "form" }
import plotly.express as px

cosine_stages = report["cosine"]["stages"]
fig = px.imshow(
    report["cosine"]["matrix"].float().numpy(),
    x=cosine_stages,
    y=cosine_stages,
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    text_auto=".2f",
    title="Cosine similarity between stage directions",
)
fig.update_layout(height=650, width=850)
display_plotly_figure(fig, "it-stage-cosine")

## 4. Steering: a readout after the final norm, and a causal edit at block 10

The target gap is $\mathrm{logit}(\text{Fruit}) - \mathrm{logit}(\text{Color})$; the prompt asks about `orange` and
the model answers `Color`, so it starts negative.

**At `unembed.hook_in`** each unit direction is added at fixed scales after the final norm. Nothing downstream can
transform it there, so the gap change is exactly linear in scale: this column measures how well each direction aligns
with the difference of the two target unembeddings, not what it does to the computation.

**At block 10** each stage is recomputed from the states at that hook and added there, at fractions of the stage's own
group separation at that site (residual norms there are hundreds, so fixed absolute scales would move little). This is
the causal column: the edit passes through every later layer.

In [7]:
# @title 4: Steering tables { display-mode: "form" }
# the gap is logit(Fruit) - logit(Color): a rise moves the answer toward Fruit
GAP_LEGEND = ((POSITIVE_COLOUR, "gap rises (toward Fruit)"), (NEGATIVE_COLOUR, "gap falls (toward Color)"))


def display_sweep(per_stage, key, label, *, title, caption):
    """One row per stage (in report order), one column per swept value, each cell the signed gap change."""
    values = [rec[key] for rec in next(iter(per_stage.values()))]
    display_notebook_table(
        [(name, *(Signed(rec["gap_delta"]) for rec in recs)) for name, recs in per_stage.items()],
        ["stage", *(label(v) for v in values)],
        title=title,
        caption=caption,
        legend=GAP_LEGEND,
    )


display_sweep(
    report["steering"],
    "scale_factor",
    lambda v: f"scale {v:g}",
    title=f"Readout at {report['steering_site']}: gap change per scale",
    caption="Each stage's unit direction added after the final norm: linear in scale, so a readout of alignment.",
)
causal = report["causal_steering"]
display_sweep(
    causal["efficacy"],
    "fraction",
    lambda v: f"{v:g}x separation",
    title=f"Causal steering at {causal['hook']}: gap change per fraction of each stage's separation",
    caption="Each stage recomputed from the states at this hook and added there; every later layer transforms it.",
)

stage,scale 10,scale 25,scale 50,scale 100
target_token_embed_difference,+12.50,+31.25,+63.00,+125.94
embed_mean_difference,+2.75,+7.25,+14.75,+29.50
answer_state_mean_difference,-0.50,-1.00,-1.75,-3.75
context_state_mean_difference,+0.25,+1.00,+2.00,+4.00
projected_context_mean_difference,+0.25,+0.75,+1.75,+3.75
paired_residual_unweighted,+0.00,+0.50,+0.75,+1.75
store_context_paired_rejection,+0.25,+0.50,+1.50,+3.00
embed_paired_rejection,+2.50,+6.75,+13.25,+26.75
store_answer_paired_rejection,-0.75,-1.25,-2.25,-4.75


stage,0.25x separation,0.5x separation,1x separation,2x separation
target_token_embed_difference,+1.00,+2.75,+8.50,+34.88
embed_mean_difference,+1.00,+2.25,+4.75,+2.53
answer_state_mean_difference,+1.00,+2.50,+6.50,+0.09
context_state_mean_difference,+0.50,+1.50,+6.75,+16.25
projected_context_mean_difference,+0.50,+1.25,+5.75,+19.00
paired_residual_unweighted,-1.00,+0.25,+5.25,+15.00
store_context_paired_rejection,-1.00,+0.25,+5.50,+16.12
embed_paired_rejection,+1.50,+2.75,+6.00,+9.88
store_answer_paired_rejection,+0.50,+1.50,+3.50,+2.50


## 5. Which features each stage's attribution graph selects

For each stage, one attribution graph with the stage's direction as its target, its top features (layers 10 and up,
ranked by absolute signed influence), and the gap change when those features are intervened on. A feature is a
transcoder latent `(layer, position, index)`: its **encoder** vector reads the residual stream at the transcoder's input
hook of that layer, and its **decoder** vector writes back at the output hook, which is where the intervention acts.
The position indexes the chat-rendered prompt, decoded beside it. Explanations come from the Neuronpedia instance the
experiment configures, where one is available for the feature.

In [8]:
# @title 5: Per-stage top features { display-mode: "form" }
import yaml
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from interpretune.harness.notebook_ui import resolve_feature_explanations
from interpretune.harness.sessions import resolve_model_spec

graphs = report.get("stage_graphs") or {}
assert graphs, "the experiment ran without stage_graphs: enable LATENT_DYNAMICS.stage_graphs to fill this section"
spec = resolve_model_spec(payload["MODEL"]["family"], payload["MODEL"]["variant"])
prompt_ids = next(iter(graphs.values()))["rendered_prompt_token_ids"]
tokenizer = AutoTokenizer.from_pretrained(spec.model_name)

# the transcoder set names the hooks its encoders read and its decoders write
tc_repo_owner, tc_repo_name, tc_subfolder = spec.transcoder_set.split("/", 2)
with open(hf_hub_download(f"{tc_repo_owner}/{tc_repo_name}", "config.yaml", subfolder=tc_subfolder)) as f:
    tc_hooks = yaml.safe_load(f)

np_cfg = payload.get("NEURONPEDIA") or {}
np_base_url = np_cfg["local_webapp_url"] if np_cfg.get("use_localhost") else "https://www.neuronpedia.org"
# features as plain ints, so the explanation lookup keys match
shown = {
    name: [(tuple(int(v) for v in fid), float(score)) for fid, score in zip(g["feature_ids"], g["feature_scores"])][
        :TOP_FEATURES_SHOWN
    ]
    for name, g in graphs.items()
}
shown_pairs = {(layer, index) for feats in shown.values() for (layer, _, index), _ in feats}
explanations = resolve_feature_explanations(
    model_id=spec.neuronpedia_model,
    source_set=spec.neuronpedia_set,
    feature_tuples=sorted(shown_pairs),
    base_url=np_base_url,
)

feature_rows = []
for name, feats in shown.items():
    for rank, ((layer, pos, index), score) in enumerate(feats, start=1):
        feature_rows.append(
            (
                name if rank == 1 else "",
                Signed(graphs[name]["gap_delta"]) if rank == 1 else "",
                rank,
                layer,
                f"{pos} {tokenizer.decode([prompt_ids[pos]])!r}",
                index,
                Signed(score, "+.3g"),
                explanations.get((layer, index), ""),
                f"blocks.{layer}.{tc_hooks['feature_input_hook']}",
                f"blocks.{layer}.{tc_hooks['feature_output_hook']}",
            )
        )
display_notebook_table(
    feature_rows,
    [
        "stage",
        "gap change",
        "rank",
        "layer",
        "position",
        "feature",
        "influence",
        "explanation",
        "encoder reads",
        "decoder writes",
    ],
    title=f"Top {TOP_FEATURES_SHOWN} features of each stage's attribution graph ({spec.neuronpedia_set})",
    caption=f"Gap change: intervening on the stage's selected features. Explanations from {np_base_url}: "
    f"{len(explanations)} of {len(shown_pairs)} features have one there.",
    legend=SIGN_LEGEND,
)

stage,gap change,rank,layer,position,feature,influence,explanation,encoder reads,decoder writes
target_token_embed_difference,+91.75,1,25,27 '\n',4973,+3.46e-08,,blocks.25.mlp.hook_in,blocks.25.hook_mlp_out
,,2,25,27 '\n',1316,-2.3e-08,,blocks.25.mlp.hook_in,blocks.25.hook_mlp_out
,,3,16,17 ' Color',13701,-2.03e-08,,blocks.16.mlp.hook_in,blocks.16.hook_mlp_out
,,4,23,27 '\n',3503,+1.28e-08,,blocks.23.mlp.hook_in,blocks.23.hook_mlp_out
,,5,16,7 ' color',13701,-1.1e-08,,blocks.16.mlp.hook_in,blocks.16.hook_mlp_out
embed_mean_difference,+51.72,1,20,27 '\n',1805,+3.54e-08,,blocks.20.mlp.hook_in,blocks.20.hook_mlp_out
,,2,25,27 '\n',1316,-1.74e-08,,blocks.25.mlp.hook_in,blocks.25.hook_mlp_out
,,3,16,17 ' Color',13701,-1.63e-08,,blocks.16.mlp.hook_in,blocks.16.hook_mlp_out
,,4,25,27 '\n',4973,+1.28e-08,,blocks.25.mlp.hook_in,blocks.25.hook_mlp_out
,,5,25,27 '\n',765,+1.05e-08,,blocks.25.mlp.hook_in,blocks.25.hook_mlp_out


## 6. Projections

Two UMAP projections. The first places every probe example's states (answer position, context, projected and final
store state) in one plane, coloured by state and marked by sense group, so the path each example takes from context to
answer is visible. The second places the stage directions themselves.

In [9]:
# @title 6: Trajectory and direction projections { display-mode: "form" }
import pandas as pd

trajectory = report["projections"]["trajectory"]
fig = px.scatter(
    pd.DataFrame(trajectory["records"]),
    x="x",
    y="y",
    color="state",
    symbol="group",
    hover_data=["example_index"],
    title=f"Per-example latent states ({trajectory['method']}, {trajectory['backend']})",
)
fig.update_layout(height=550, width=850)
display_plotly_figure(fig, "it-trajectory-projection")

directions = report["projections"]["directions"]
fig = px.scatter(
    pd.DataFrame(directions["records"]),
    x="x",
    y="y",
    color="basis",
    symbol="mode",
    text="stage",
    title=f"Stage directions ({directions['method']}, {directions['backend']})",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=550, width=850)
display_plotly_figure(fig, "it-direction-projection")

## What to look for

- **Three families, not one direction.** The embedding, answer-state and context-state stages form separate blocks in
  the cosine matrix: the same concept is carried along unrelated directions at different points in the computation.
- **Readout is not causation.** The final-site column ranks the embedding stages far above the store stages, because
  they are built from (or close to) the target unembeddings. At block 10 the store stages produce effects of the same
  order as the embedding stages once each is scaled by its own separation.
- **Shared features, opposite effects.** The answer-state and context-state graphs share their mid-layer features at
  the answer position and differ in their late features, which is where the sign of their effect is decided.

The measured numbers for this configuration are discussed on
[issue #971](https://github.com/speediedan/interpretune/issues/971).